# Skalierung & Hyperparameter-Tuning

![Illustration zu Skalierung und Hyperparameter-Tuning](https://i.ytimg.com/vi/b0L_2jKEbA4/maxresdefault.jpg)

In diesem Notebook siehst du an einem kurzen Beispiel, wie du deine Daten skalierst und die Hyperparameter deiner Modelle mit Grid Search oder Randomized Search tunst. Dabei nutzt du die Kreuzvalidierung, um deine Modelle zuverlässiger zu evaluieren, als es eine einzelne Aufteilung in Trainings- und Testdaten erlaubt.

Nutz die rohe Titanic-CSV aus diesem Repository, um Feature-Skalierung und Hyperparameter-Suchen zu vergleichen. Imputation und Kodierung bleiben **innerhalb jeder Pipeline**, damit die Validierungszeilen die Vorverarbeitung nicht beeinflussen.

Die ältere `titanic_preprocessed.csv` bleibt zum Nachschlagen verfügbar, wird hier aber nicht verwendet, weil ihre Vorverarbeitung vor der Aufteilung in diesem Notebook stattfand. Wir behalten dieselben sechs Ausgangs-Features (`Age`, `SibSp`, `Parch`, `Fare`, `Sex`, `Embarked`); geändert hat sich, wann die Vorverarbeitung gefittet wird, nicht welche Informationen die Modelle bekommen.

**Nächstes Notebook:** Hier wählt die Kreuzvalidierung das Modell innerhalb des Trainingsdatensatzes aus. Die Lösung der Aufgabe zu Herzerkrankungen ergänzt einen separaten Validierungsdatensatz, um Kandidaten nach dem Hyperparameter-Tuning auf den Trainings-Folds zu vergleichen. Beide Protokolle heben den Testdatensatz für eine abschließende Evaluation auf; ein separater Validierungsdatensatz ist nicht immer nötig.

## Lernziele

Am Ende dieses Notebooks kannst du:

- numerische Features mit Standardisierung (`StandardScaler`) und Min-Max-Skalierung (`MinMaxScaler`) skalieren und erklären, wann welche davon nützlich ist.
- Vorverarbeitung und ein Modell in einer einzigen `Pipeline` von scikit-learn kombinieren.
- Modelle mit K-facher Kreuzvalidierung evaluieren, statt dich auf eine einzige Aufteilung in Trainings- und Testdaten zu verlassen.
- Hyperparameter mit `GridSearchCV` und `RandomizedSearchCV` tunen und die Trade-offs zwischen beiden vergleichen.

In [ ]:
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    # StratifiedKFold,  # für die Alternative unten einkommentieren
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 10
# 5 Folds für alle Vergleiche. Bei Klassifikatoren nutzt scikit-learn automatisch
# stratifizierte Folds (Klassenanteile bleiben erhalten), allerdings ohne Mischen.
cv = 5
# Alternative: Folds explizit mischen und stratifizieren
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)

## Der Titanic-Datensatz

Die mitgelieferte Datei `data/titanic.csv` enthält **891 Passagiereinträge mit Labels**: eine Zeile pro Person und 12 Spalten. Wir sagen `Survived` vorher, ein binäres Ergebnis: **0 = nicht überlebt; 1 = überlebt**. Das ist eine historische Klassifikationsaufgabe, kein System, um zu entscheiden, wer Hilfe bekommen sollte.

Die CSV folgt dem öffentlichen [Datenwörterbuch des Titanic-Wettbewerbs](https://www.kaggle.com/competitions/titanic/data):

| Spalte | Bedeutung | Rolle in diesem Notebook |
| --- | --- | --- |
| `PassengerId` | Passagierkennung | Ausgeschlossen |
| `Survived` | Überlebensstatus, 0 oder 1 | Vorherzusagende Zielvariable |
| `Pclass` | Ticketklasse: 1., 2. oder 3. | Aus diesem begrenzten Beispiel ausgeschlossen |
| `Name` | Passagiername | Ausgeschlossen |
| `Sex` | Erfasstes Geschlecht: female oder male | Kategoriale Eingabe |
| `Age` | Alter in Jahren; auch Werte mit Nachkommastellen kommen vor | Numerische Eingabe |
| `SibSp` | Anzahl der Geschwister/Ehepartnerinnen und Ehepartner an Bord | Numerische Eingabe |
| `Parch` | Anzahl der Eltern/Kinder an Bord | Numerische Eingabe |
| `Ticket` | Ticketnummer | Ausgeschlossen |
| `Fare` | Fahrpreis | Numerische Eingabe |
| `Cabin` | Kabinennummer | Ausgeschlossen |
| `Embarked` | Hafen: C = Cherbourg, Q = Queenstown, S = Southampton | Kategoriale Eingabe |

Aus diesen Einträgen mit Labels erstellen wir unsere **eigene Aufteilung in Trainings- und Testdaten**. Die separate Testdatei des Wettbewerbs ohne Labels und eine Einreichung für das Leaderboard brauchen wir nicht.

Die Labels sind unbalanciert: 549 Reisende haben nicht überlebt und 342 haben überlebt. Deshalb vergleichen wir die Accuracy später mit einer Mehrheitsklassen-Baseline. Sieh dir zuerst ein paar Zeilen und die fehlenden Werte an. In dieser Datei hat `Age` 177 fehlende Werte, `Cabin` 687 und `Embarked` 2. Fehlende Werte werden später innerhalb der Pipeline behandelt; wir füllen sie nicht mit Kennzahlen aus dem gesamten Datensatz.

In [ ]:
# Rohdaten laden, damit die Vorverarbeitung innerhalb jedes Trainings-Folds gelernt werden kann.
df = pd.read_csv("data/titanic.csv")
df.head()

In [ ]:
# Fehlende Werte in jeder Spalte zählen, bevor der Umgang damit festgelegt wird.
df.isnull().sum()

## Aufteilung in Trainings- und Testdaten

`X` enthält die **Features**; `y` enthält das **Ziel-Label**, das wir vorhersagen wollen. Jede Zeile in `X` und `y` bezieht sich auf dieselbe Person.

### Warum diese sechs Eingaben?

Wir behalten vier numerische Eingaben (`Age`, `SibSp`, `Parch`, `Fare`) und zwei kategoriale Eingaben (`Sex`, `Embarked`). Sie bieten eine kleine Mischung aus Altersangaben, Anzahlen, Fahrpreisen und Kategorien, um Imputation, Skalierung und Kodierung zu üben. Das ist eine **didaktische Entscheidung**, keine Behauptung, dass dies die besten Prädiktoren sind.

Warum nicht alle Spalten verwenden?

- `Survived` ist die Antwort. Sie in `X` aufzunehmen, würde die Zielvariable verraten.
- `PassengerId` ist eine Kennung, keine aussagekräftige Messung zur Person.
- `Name` und `Ticket` brauchen Feature Engineering, bevor sie hier nützlich sein können; ihr Rohtext ist keine direkt verwendbare numerische Eingabe.
- `Cabin` fehlt größtenteils und erfordert zusätzliche Entscheidungen über die Darstellung und das Fehlen von Werten.
- `Pclass` ist ebenfalls ein plausibler Prädiktor. Er wird weggelassen, um beim selben Beispiel mit sechs Eingaben und beim Input Contract zu bleiben, den Notebook 3 wiederverwendet, **nicht weil er irrelevant ist**.

Weitere Features hinzuzufügen ist eine zulässige Erweiterung: Aktualisiere die Vorverarbeitung und vergleich per Kreuzvalidierung auf den Trainingsdaten, nicht mit dem Test-Score. Wenn du dieses Modell exportierst, aktualisiere auch das Eingabeschema von Notebook 3.

In [ ]:
# Label von den sechs Eingaben trennen; Survived nie an den Klassifikator geben.
y = df["Survived"]
X = df[["Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]]

In [ ]:
# Ausgewählte Spalten untersuchen: Survived sollte nicht mehr enthalten sein.
X.head(2)

### Testdatensatz beiseitelegen

`test_size=0.2` legt etwa 20% der Reisenden für die **abschließende Evaluation** beiseite; die übrigen 80% dienen dem Training und der Kreuzvalidierung. `random_state=42` macht diese Aufteilung wiederholbar. `stratify=y` erhält die Überlebensanteile in beiden Datensätzen näherungsweise.

Die Trainings-Folds leiten die Wahl von Skalierung und Hyperparametern. Wähl keine Konfiguration anhand der Accuracy auf den zurückgehaltenen Testdaten aus, um dann denselben Score als unabhängige Evaluation zu berichten.

In [ ]:
# Features und Labels gemeinsam aufteilen, damit jede Person das richtige Label behält.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

---

## Feature-Skalierung

Oft haben die Features deines Modells unterschiedliche Einheiten, und damit haben die Variablen auch unterschiedliche Skalen. Während manche Modelltypen (z. B. baumbasierte Modelle wie Entscheidungsbaum oder Random Forest) im Allgemeinen keine Skalierung der numerischen Eingabevariablen brauchen, können viele Machine-Learning-Algorithmen, darunter zum Beispiel Algorithmen mit Distanzmaßen (z. B. KNN, SVM), besser abschneiden, wenn numerische Features passend skaliert sind; eine Verbesserung ist nicht garantiert.

Zwei gängige Verfahren sind die **Min-Max-Skalierung** (in der Abbildung als Normalization bezeichnet) und die **Standardisierung**. Der `Normalizer` von scikit-learn skaliert dagegen jede Zeile auf die Einheitsnorm; das ist nicht dasselbe wie die Min-Max-Skalierung.

Weitere Infos findest du in [diesem Leitfaden zu Skalierungstransformationen](https://machinelearningmastery.com/standardscaler-and-minmaxscaler-transforms-in-python/).

![Originaldaten im Vergleich mit Min-Max-skalierten und standardisierten Daten](assets/normalization_vs_standardization.png)

Der ursprüngliche Datensatz in der Abbildung enthält Beobachtungen mit unterschiedlichen Wertebereichen in den beiden Features. Die Abbildung zeigt zwei gängige Skalierungsverfahren:

- **Min-Max-Skalierung** (in der Abbildung als Normalization bezeichnet): Zieh das Minimum der Trainingsdaten ab und teil dann durch die Spannweite der Trainingsdaten. Bei einem nicht konstanten Feature wird das Minimum der Trainingsdaten zu 0, das Maximum zu 1, und Werte dazwischen werden proportional abgebildet. Neue Werte können außerhalb von [0; 1] liegen.
- **Standardisierung**: Zieh den Mittelwert der Trainingsdaten ab und teil dann durch die Standardabweichung der Trainingsdaten. Nicht konstante Trainings-Features werden auf 0 zentriert und haben eine Standardabweichung der Grundgesamtheit von 1. Dadurch werden ihre Skalen vergleichbar, ihre Verteilungen aber nicht zu Normalverteilungen.

Beide Methoden erhalten die Reihenfolge und die Form der Verteilung jedes Features und ändern dabei seine numerische Skala. Wenn die Features unterschiedlich umskaliert werden, ändern sich die Distanzen zwischen den Beobachtungen, und genau deshalb können sich die Vorhersagen von KNN ändern. Konstante Features werden gesondert behandelt; sie können keine Varianz von 1 haben.

### Standardisierung

Die Standardisierung bringt numerische Features auf eine vergleichbare Skala. Zieh zuerst den Mittelwert der Trainingsdaten ab, um die Werte um null zu zentrieren, und teil dann durch die Standardabweichung der Trainingsdaten, um ihren Abstand vom Mittelwert in Einheiten der Standardabweichung auszudrücken. Für ein nicht konstantes Feature gilt:

$$x_{scaled} = \frac{x-\mu_{train}}{\sigma_{train}}$$

Dabei gilt:

- $x$ ist der einzelne Wert, der transformiert wird.
- $\mu_{train}$ (mu) ist der **Mittelwert** der Feature-Werte in den Trainingsdaten.
- $\sigma_{train}$ (sigma) ist ihre **Standardabweichung**, die misst, wie stark die Werte um den Mittelwert streuen.

Für $m$ beobachtete Trainingswerte $x_1, \ldots, x_m$ werden diese Kennzahlen so berechnet:

$$\mu_{train} = \frac{1}{m}\sum_{i=1}^{m}x_i$$

$$\sigma_{train} = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_{train})^2}$$

Dabei ist $m$ die Anzahl der beobachteten Trainingswerte, $i$ nummeriert sie durch, und $\sum$ bedeutet, sie aufzusummieren. `StandardScaler` verwendet diese Standardabweichung der Grundgesamtheit (Division durch $m$, nicht durch $m-1$).

**Beispiel:** Wenn der Mittelwert der Trainingsdaten 50 und die Standardabweichung 10 ist, wird ein Wert von 70 zu $(70-50)/10=2$: zwei Standardabweichungen über dem Mittelwert der Trainingsdaten. Ein Wert von 50 wird zu 0 und ein Wert von 40 zu -1.

**Nach der Transformation:** Nicht konstante Trainings-Features haben Mittelwert 0 und Standardabweichung 1. Konstante Spalten werden nach dem Zentrieren zu 0. Die Standardisierung erhält die Form der Verteilung; sie macht die Daten nicht normalverteilt.

Fitte den Scaler nur auf den Trainingsdaten; innerhalb der Kreuzvalidierung fittest du ihn in jedem Trainings-Fold separat. Verwende diese Kennzahlen dann für Validierungs- oder Testdaten wieder, deren transformierter Mittelwert und transformierte Standardabweichung nicht 0 und 1 sein müssen.

[`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) ist empfindlich gegenüber Ausreißern.

Diesen `StandardScaler` verwendet `scaling_transformation` oben bereits.

### Min-Max-Skalierung

Die Min-Max-Skalierung (manchmal Normalisierung genannt) skaliert ein Feature auf einen gewählten Wertebereich um, standardmäßig 0 bis 1. Zieh das Minimum der Trainingsdaten ab und teil dann durch die Spannweite der Trainingsdaten. Für ein nicht konstantes Feature gilt:

$$x_{scaled} = \frac{x-x_{min,train}}{x_{max,train}-x_{min,train}}$$

Dabei gilt:

- $x$ ist der Wert, der transformiert wird.
- $x_{min,train}$ ist der kleinste beobachtete Trainingswert dieses Features.
- $x_{max,train}$ ist der größte beobachtete Trainingswert. Ihre Differenz ist die Spannweite der Trainingsdaten.

**Beispiel:** Bei einem Minimum der Trainingsdaten von 10 und einem Maximum von 30 wird ein Wert von 20 zu $(20-10)/(30-10)=0.5$. Ein neuer Wert von 40 wird zu 1,5: Berechne Minimum und Maximum nicht auf den Testdaten neu, um ihn in [0; 1] zu zwingen.

[`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) lernt diese Grenzen aus den Trainingsdaten und verwendet sie für neue Beobachtungen wieder. Nicht konstante Trainings-Features werden auf [0; 1] abgebildet; konstante Trainings-Features werden mit den Standardeinstellungen auf 0 abgebildet. Er entfernt keine Ausreißer, und extreme Trainingswerte können die meisten anderen Werte in einen kleinen Teil des Wertebereichs zusammendrücken.

### Wie die Vorverarbeitungs-Pipeline funktioniert

Modelle brauchen numerische Eingaben ohne fehlende Werte. Ein `ColumnTransformer` schickt die beiden Arten von Spalten durch unterschiedliche Schritte und kombiniert dann ihre Ausgaben:

| Eingabespalten | Schritte der Reihe nach | Warum? |
| --- | --- | --- |
| Numerisch | Fehlende Werte mit dem Median der Trainingsdaten (mittlerer Wert) füllen, dann optional skalieren | Zeilen mit fehlendem Alter behalten und Einheiten vergleichbar machen |
| Kategorial | Fehlende Werte mit dem Modus der Trainingsdaten (häufigste Kategorie) füllen, dann per One-Hot Encoding kodieren | Text durch Indikatorspalten ersetzen, ohne eine numerische Reihenfolge aufzuzwingen |

Das One-Hot Encoding erstellt für jede gelernte Kategorie eine 0/1-Spalte: Eine als female erfasste Person hat `Sex_female=1` und `Sex_male=0`, wenn beide Kategorien gelernt wurden. `handle_unknown="ignore"` stellt eine nicht gesehene Kategorie mit lauter Nullen in ihrem Kategorienblock dar; bei der Vorhersage wird dabei keine neue Kategorie gelernt.

Eine `Pipeline` verbindet die Vorverarbeitung mit dem Klassifikator. Während der Kreuzvalidierung lernt jede Kopie Mediane, Kategorien und Skalierungskennzahlen **nur aus ihrem Trainings-Fold** und wendet sie dann unverändert auf den Validierungs-Fold an. Ohne Skalierung werden numerische Spalten nur imputiert; Imputation und Kodierung finden immer statt. Beim Erstellen der Objekte unten wird noch nichts gefittet.

[Referenz: heterogene Vorverarbeitung in scikit-learn](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html).

In [ ]:
X.head(3)

In [ ]:
# Numerische und kategoriale Eingaben an unterschiedliche Vorverarbeitungsschritte leiten.
col_scale = ["Age", "SibSp", "Parch", "Fare"]
col_encode = ["Sex", "Embarked"]

In [ ]:
# Definiere Operationen an numerische Spalten: fehlende Werte mit Median imputieren, dann standardisieren.
numeric_standardized = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Definiere Operationen an kategorischen Spalten: fehlende Werte mit der häufigsten Kategorie füllen, dann OHE.
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Festlegen, welche Operationen an welchen Spalten durchgeführt werden sollen
scaling_transformation = ColumnTransformer(
    [
        ("numeric", numeric_standardized, col_scale),       # vervollständigen + standardisieren
        ("categorical", categorical_pipeline, col_encode),  # die kategorischen vervollständigen + OHE
    ]
)

# Zum Vergleich: ohne Skalierung der numerischen Spalten
no_scaling_transformation = ColumnTransformer(
    [
        ("numeric", SimpleImputer(strategy="median"), col_scale),  # die numerischen Spalten vervollständigen
        ("categorical", categorical_pipeline, col_encode),         # die kategorischen vervollständigen + OHE
    ]
)

In [ ]:
scaling_transformation

### Aufgabe 1: Übungsrunde

Erstell dieselbe Vorverarbeitungs-Pipeline mit Min-Max-Skalierung statt Standardisierung. Fitte sie nicht vor der Kreuzvalidierung auf allen Daten.

<details>
<summary>Antwort anzeigen</summary>

```python
from sklearn.preprocessing import MinMaxScaler

numeric_normalized = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
    ]
)
normalization_transformation = ColumnTransformer(
    [
        ("numeric", numeric_normalized, col_scale),
        ("categorical", categorical_pipeline, col_encode),
    ]
)
```

`MinMaxScaler` skaliert jede numerische Spalte mit dem Minimum und Maximum um, die er aus den Trainingsdaten gelernt hat, und bildet so ihren Wertebereich in den Trainingsdaten auf 0 bis 1 ab. Imputation und kategoriale Kodierung bleiben unverändert. Die Pipeline fittet jeden gelernten Schritt innerhalb jedes Trainings-Folds separat.
</details>

In [ ]:
# Skalierung mit dem MinMaxScaler

# Versuchen, deine Daten mit dem MinMaxScaler() aus sklearn zu skalieren.
# Die Syntax ist dieselbe wie beim StandardScaler.
# Nicht vergessen: Du musst den Scaler oben in deinem Notebook importieren.

### Bonusfrage

Haben standardisierte oder Min-Max-skalierte Testwerte zwangsläufig dieselben Eigenschaften wie transformierte Trainingswerte?

Für nicht konstante Trainings-Features heißt das: ein Mittelwert von 0 und eine Standardabweichung von 1 nach der Standardisierung oder Werte zwischen 0 und 1 nach der Min-Max-Skalierung.

<details>
<summary>Antwort anzeigen</summary>

Nicht unbedingt. Der Scaler lernt seine Parameter **nur aus den Trainingsdaten** und wendet dann genau diese festen Parameter auf die Testdaten an:

- **Standardisierung**: Sie verwendet Mittelwert und Standardabweichung der Trainingsdaten wieder. Transformierte Test-Features müssen weder Mittelwert 0 noch Standardabweichung 1 haben und nicht einmal nahe daran liegen, wenn sich die Verteilung verschoben hat.
- **Min-Max-Skalierung**: Sie verwendet Minimum und Maximum der Trainingsdaten wieder. Testwerte unter diesem Minimum oder über diesem Maximum liegen außerhalb von [0; 1], möglicherweise sogar deutlich.
- **Konstante Features**: Selbst in den Trainingsdaten kann ein konstantes Feature keine Varianz von 1 haben; mit der standardmäßigen Zentrierung wird es zu 0.

Das ist zu erwarten: Verwende den gefitteten Scaler wieder, damit das Modell dieselbe Darstellung bekommt. Einen zweiten Scaler auf dem Testdatensatz zu fitten, würde zurückgehaltene Informationen nutzen und diese Darstellung verändern.
</details>

---

## Prädiktive Modellierung

Wir evaluieren die Modellleistung auf schnelle und zuverlässigere Weise mit [cross_val_score()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) aus sklearn, das die K-fache Kreuzvalidierung implementiert. Wenn wir ein Modell auf einer einzigen Aufteilung in Trainings- und Testdaten trainieren, haben wir nur ein Experiment. Können wir einem einzigen Experiment wirklich trauen?

Stell dir die [K-fache Kreuzvalidierung](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) als K Experimente vor, deren Fehler gemittelt wird. Sie ist immer noch nicht perfekt, aber vertrauenswürdiger als eine einzige Aufteilung, die zufällig ungewöhnlich gut oder schlecht ausfallen kann.

Damit stellt sich die Frage, welcher Wert für K sinnvoll ist. Übliche Werte liegen zwischen 5 und 10, und du musst auch technische Grenzen berücksichtigen: Datensatzgröße, Rechenleistung, verfügbaren Arbeitsspeicher und Zeit. Auf großen Datensätzen dauert die Kreuzvalidierung eine Weile.

![Fünffache Kreuzvalidierung mit einem anderen Validierungs-Fold in jeder Runde](assets/cross_validation.png)

Für einen fairen Vergleich verwendet jeder der folgenden Kandidaten dieselbe fünffache Aufteilung (`cv=5`; bei Klassifikatoren stratifiziert scikit-learn automatisch) und Accuracy als Metrik. Die Stratifizierung erhält die Klassenanteile näherungsweise; sie verhindert aber keine Datenlecks zwischen verwandten Reisenden wie Familienmitgliedern.

Der Trainingsdatensatz wird mit `cv=5` in **5 Folds** aufgeteilt. In jeder Runde:

- Ein Fold wird für die **Validierung** zurückgehalten.
- Auf den anderen vier Folds wird eine neue Kopie von Vorverarbeitung und Modell trainiert.
- Diese Kopie macht Vorhersagen für den Validierungs-Fold und liefert einen Accuracy-Score.

Jeder Fold wird einmal zur Validierung verwendet, deshalb gibt `cross_val_score` fünf Scores zurück. Ihr Mittelwert fasst die Leistung zusammen; ihre Standardabweichung beschreibt die Streuung und ist **kein Konfidenzintervall**, weil sich die Trainings-Folds überschneiden. Diese Validierungs-Folds stammen nur aus `X_train`; `X_test` liegt weiterhin beiseite.

**Wenn du die Folds oder das Netz der Suche änderst, starte den Kernel neu und führ alle Zellen erneut aus** (auch die Aufgaben, die du löst). Gespeicherte Ausgaben können frühere Einstellungen widerspiegeln; vergleich Ergebnisse aus demselben Durchlauf.

### Baseline: immer die Mehrheitsklasse vorhersagen

Bevor du KNN oder SGD ausprobierst, leg eine einfache Referenz fest. `DummyClassifier(strategy="most_frequent")` ignoriert die Features und sagt in jedem Trainings-Fold immer das häufigste Label vorher. Evaluiere ihn wie die anderen Modelle mit denselben Folds und mit Accuracy als Metrik; berechne die Mehrheitsklasse nicht aus dem gesamten Datensatz oder dem Testdatensatz.

Ein Modell sollte diese Referenz übertreffen, um seine Komplexität zu rechtfertigen. Sie zu schlagen, belegt aber noch keinen Nutzen in der Praxis. [Referenz: DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html).

In [ ]:
# Mehrheitslabel unabhängig innerhalb jedes Trainings-Folds lernen.
baseline_scores = cross_val_score(
    DummyClassifier(strategy="most_frequent"),  # DummyClassifier sagt einfach die häufigste Kategorie vorher
    X_train,
    y_train,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
)
# Scores der Folds aufbewahren, damit spätere Beispiele verglichen werden können, ohne die CV erneut auszuführen.
cv_scores = {"Majority-class baseline": baseline_scores}
print(f"Baseline mean CV accuracy: {baseline_scores.mean():.3f}")

### KNN-Klassifikator: unskalierte Daten

K-Nearest Neighbors (KNN) findet mit einem Distanzmaß die ähnlichsten Reisenden in den Trainingsdaten und kombiniert dann deren Labels, um das Überleben vorherzusagen. Große numerische Wertebereiche können diese Distanz dominieren: Ein Unterschied im Fahrpreis von 100 kann einen Altersunterschied von 10 überwiegen. Die Skalierung verändert, wie die numerischen Eingaben beitragen.

Hier umfasst „unskaliert“ trotzdem die Behandlung fehlender Werte und die kategoriale Kodierung. `cross_val_score` trainiert für jeden Fold eine neue Pipeline und gibt **Accuracy-Werte der Validierung** zurück, nicht den Score auf dem beiseitegelegten Testdatensatz.

In [ ]:
# Komplette Pipeline aus Vorverarbeitung und Modell innerhalb jedes CV-Folds evaluieren.
knn_classifier = Pipeline(
    [
        ("preprocessing", no_scaling_transformation),
        ("knn", KNeighborsClassifier()),
    ]
)
scores = cross_val_score(
    knn_classifier, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["KNN - unscaled"] = scores
print(f"Mean accuracy (unscaled): {scores.mean():.3f}")
print(f"Fold standard deviation (unscaled): {scores.std():.3f}")

In [ ]:
scores

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# Ein Balken pro Validierungs-Fold; passt sich automatisch an, wenn sich cv ändert.
sns.barplot(x=range(1, len(scores) + 1), y=scores, ax=ax)
ax.axhline(scores.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="KNN: unscaled inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Bildbeschreibung:
Der Plot mit dem Titel "KNN: unscaled inputs" zeigt einen Accuracy-Balken pro Validierungs-Fold (fünf bei der aktuellen Konfiguration). Die horizontale Linie ist ihr Mittelwert. Vergleich die Balkenhöhen, um die Streuung zu untersuchen, und lies die aktuelle Ausgabe, statt feste Werte über Durchläufe oder Paketversionen hinweg zu erwarten.
Ende der Bildbeschreibung

### KNN-Klassifikator: skalierte Daten

In [ ]:
# Mit Kreuzvalidierung trainieren und evaluieren; die zurückgehaltenen Testdaten werden nicht verwendet.
# Dieselben Folds und Vorverarbeitungsentscheidungen verwenden; nur die numerische Skalierung ändern.
knn_classifier_scaled = Pipeline(
    [
        ("scaling_trans", scaling_transformation),
        ("knn", KNeighborsClassifier()),
    ]
)
scores_scaled = cross_val_score(
    knn_classifier_scaled, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["KNN - standardized"] = scores_scaled
print(f"Mean accuracy (scaled): {scores_scaled.mean():.3f}")
print(f"Fold standard deviation (scaled): {scores_scaled.std():.3f}")
print(
    f"Accuracy difference: {100 * (scores_scaled.mean() - scores.mean()):+.1f} percentage points"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# Ein Balken pro Validierungs-Fold; passt sich automatisch an, wenn sich cv ändert.
sns.barplot(x=range(1, len(scores_scaled) + 1), y=scores_scaled, ax=ax)
ax.axhline(scores_scaled.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="KNN: standardized inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Vergleich die mittlere Accuracy mit und ohne Skalierung sowie die Streuung über die Folds. Skalierung kann die Leistung verbessern, aber weder die Größe noch die Richtung der Änderung ist garantiert. Eine Änderung von 0,70 auf 0,75 entspricht 5 Prozentpunkten, nicht einem relativen Anstieg um 5%. Um den Effekt der Skalierung deutlicher zu zeigen, sehen wir uns einen weiteren Klassifikator an, der noch empfindlicher darauf reagiert.

### SGD-Klassifikator: unskalierte Daten

Der [SGD-Klassifikator](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) (Klassifikator mit stochastischem Gradientenabstieg) implementiert eine Vielzahl von Klassifikatoren. Welches Modell er trainiert, steuert der Parameter `loss`: Standardmäßig trainiert er eine lineare Support Vector Machine (SVM). Wir verwenden ihn hier, um die Bedeutung der Skalierung hervorzuheben, und setzen `loss="log_loss"`, sodass das Modell zu einer logistischen Regression wird, die mit stochastischem Gradientenabstieg trainiert wird.

`max_iter=10000` ist eine Obergrenze für die Durchläufe über die Trainingsdaten, kein Versprechen, alle davon auszuführen. `random_state=RSEED` macht sein stochastisches Training wiederholbar. Im nächsten Beispiel ändert die Pipeline die numerische Skalierung, während Modelleinstellungen und Validierungs-Folds gleich bleiben.

In [ ]:
# Unskalierter Vergleich: Imputation und Kodierung finden trotzdem innerhalb jedes Folds statt.
sgd_classifier = Pipeline(
    [
        ("preprocessing", no_scaling_transformation),
        # log_loss trainiert einen linearen probabilistischen Klassifikator per SGD.
        ("sgd", SGDClassifier(random_state=RSEED, loss="log_loss", max_iter=10000)),
    ]
)
scores = cross_val_score(
    sgd_classifier, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["SGD - unscaled"] = scores
print(f"Mean accuracy (unscaled): {scores.mean():.3f}")
print(f"Fold standard deviation (unscaled): {scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# Ein Balken pro Validierungs-Fold; passt sich automatisch an, wenn sich cv ändert.
sns.barplot(x=range(1, len(scores) + 1), y=scores, ax=ax)
ax.axhline(scores.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="SGD: unscaled inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Sieh dir an, wie die Validierungs-Accuracy über die verschiedenen Folds variiert. Größere Werte bedeuten eine bessere Leistung; nutz die aktuellen Ergebnisse, statt einen festen Wertebereich anzunehmen.

### SGD-Klassifikator: skalierte Daten

In [ ]:
# Die Standardisierung wird auf jedem Trainings-Fold gefittet, nicht auf den Validierungszeilen.
sgd_classifier_scaled = Pipeline(
    [
        ("scaling_trans", scaling_transformation),
        ("sgd", SGDClassifier(random_state=RSEED, loss="log_loss", max_iter=10000)),
    ]
)
scores_scaled = cross_val_score(
    sgd_classifier_scaled, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["SGD - standardized"] = scores_scaled
print(f"Mean accuracy (scaled): {scores_scaled.mean():.3f}")
print(f"Fold standard deviation (scaled): {scores_scaled.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# Ein Balken pro Validierungs-Fold; passt sich automatisch an, wenn sich cv ändert.
sns.barplot(x=range(1, len(scores_scaled) + 1), y=scores_scaled, ax=ax)
ax.axhline(scores_scaled.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="SGD: standardized inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Vergleich vor und nach der Skalierung sowohl die mittlere Validierungs-Accuracy als auch ihre Streuung über die Folds. Skalierung kann der Optimierung per SGD helfen, aber eine bessere Accuracy und eine geringere Streuung sind nicht garantiert.

### Aufgabe 2: Übungsrunde

Probier den KNN-Klassifikator mit den Min-Max-skalierten Daten aus.

<details>
<summary>Antwort anzeigen</summary>

Verwende den Normalisierungs-Transformer aus Aufgabe 1 in einer `Pipeline` wieder, genau wie beim standardisierten KNN-Modell, und evaluiere sie dann per Kreuzvalidierung:

```python
knn_normalized = Pipeline(
    [("scaling_trans", normalization_transformation), ("knn", KNeighborsClassifier())]
)
scores_norm = cross_val_score(
    knn_normalized, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
print("Score (normalized):", round(scores_norm.mean(), 2))
cv_scores["KNN - min-max"] = scores_norm
```

Beide Scaler verändern Distanzen, müssen aber keine ähnlichen Ergebnisse liefern. Vergleich sie auf denselben Validierungs-Folds; keiner von beiden ist grundsätzlich besser.
</details>

In [ ]:
# Modell mit den Daten trainieren, die du in Aufgabe 1 mit dem MinMaxScaler skaliert hast.

# Die Syntax ist dieselbe wie beim StandardScaler.

### Skalierungsvergleich auf einen Blick

Sammle die bisherigen Scores der Folds in einer Tabelle. Der Zugewinn ist die Differenz zur Baseline in **Prozentpunkten**, kein relativer Prozentwert. Wenn du Aufgabe 2 mit der Antwort oben gelöst hast, erscheint auch das Min-Max-Ergebnis. Hier werden keine weiteren Modelle trainiert.

Welches Modell übertrifft die Baseline am deutlichsten? Wie wirkt sich die Skalierung bei KNN im Vergleich zu SGD aus? Lies die Standardabweichung über die Folds als Streuung, nicht als Konfidenzintervall oder als Beleg für einen statistisch signifikanten Unterschied. Diese Tabelle vergleicht nicht getunte Modelle; die KNN-Suchen unten sind eine separate Demonstration.

In [ ]:
# Gespeicherte Scores der Folds zusammenfassen; diese Zelle trainiert kein Modell neu.
cv_comparison = pd.DataFrame(
    {
        name: {
            "mean_cv_accuracy": fold_scores.mean(),
            "fold_standard_deviation": fold_scores.std(),
            "gain_vs_baseline_pp": 100 * (fold_scores.mean() - baseline_scores.mean()),
        }
        for name, fold_scores in cv_scores.items()
    }
).T
cv_comparison.round(3)

---

## Hyperparameter-Tuning

Hyperparameter sind Einstellungen, die wir vor dem Training festlegen, etwa die Anzahl der Nachbarn bei KNN; gelernte Parameter werden aus den Trainingsdaten gewonnen. Die meisten Modelle haben viele Hyperparameter, und die besten Werte unterscheiden sich von Datensatz zu Datensatz. Dasselbe gilt für die Regularisierungs-Hyperparameter, die wir vorher gesehen haben und die meist durch Ausprobieren gewählt werden. Wie finden wir also die Parameterwerte, die für unsere Daten am besten funktionieren?

### GridSearchCV

Das [Grid Search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) ist eine Tuning-Methode, die nach den besten Hyperparameterwerten sucht. Sie führt mit Kreuzvalidierung eine vollständige Suche über einen vordefinierten Parameterraum durch (daher das Suffix **CV**) und evaluiert jede mögliche Kombination im Suchraum, um die beste zu finden und zurückzugeben.

Das wird zeitaufwendig, wenn es viele Hyperparameter gibt und der Suchraum groß ist. In der veranschaulichenden Abbildung unten führt GridSearchCV mit k=5 und 2 Parametern, die 2 bzw. 3 Werte annehmen (6 Kombinationen), 30 Modellierungsschritte aus, nur um die besten Werte für diese beiden Parameter zu finden.

![Das Grid Search evaluiert sechs Parameterkombinationen mit fünffacher Kreuzvalidierung](assets/grid_search_cv.png)

Der Prozess evaluiert mehrere Kombinationen von Hyperparametern, um die Modellkonfiguration mit der besten Leistung zu finden.

In der Abbildung (nicht im Code unten) werden zwei Hyperparameter betrachtet:

- Parameter **a** mit den möglichen Werten **1, 2 und 3**
- Parameter **b** mit den möglichen Werten **1 und 2**

Das ergibt **6 mögliche Hyperparameterkombinationen**.

Für jede Kombination wird eine **5-fache Kreuzvalidierung** durchgeführt:

- Die Trainingsdaten werden in 5 Folds aufgeteilt
- Das Modell wird 5-mal trainiert und validiert
- Die Leistungs-Scores werden gemittelt, um einen Kreuzvalidierungs-Score für diese Parameterkombination zu erhalten

Dieser Prozess wird für alle 6 Kombinationen wiederholt.

Insgesamt werden **30 Trainingsdurchläufe der Kreuzvalidierung ausgeführt (6 × 5)**. Mit der Standardeinstellung `refit=True` wird die beste Konfiguration anschließend noch einmal auf allen Trainingsdaten trainiert, die der Suche übergeben wurden.

Die Parameterkombination mit der besten durchschnittlichen Validierungsleistung wird als endgültige Modellkonfiguration ausgewählt.

#### Suche konfigurieren und beobachten

Die Namen von Hyperparametern in einer Pipeline haben die Form `step__parameter`: `knn__n_neighbors` ändert `n_neighbors` im Schritt mit dem Namen `knn`.

| KNN-Einstellung | Bedeutung |
| --- | --- |
| `n_neighbors` | Wie viele nahe Reisende aus den Trainingsdaten eine Stimme abgeben |
| `weights="uniform"` / `"distance"` | Gleich gewichtete Stimmen / mehr Einfluss für nähere Nachbarn |
| `p=1` / `2` / `3` | Manhattan- / euklidische / eine andere Minkowski-Distanz |

Das Netz unten hat **5 x 2 x 3 = 30 Kombinationen**. Mit fünf Folds führt das Grid Search **150 Trainingsdurchläufe der Kreuzvalidierung** aus, dann einen zusätzlichen Trainingsdurchlauf des Gewinners auf ganz `X_train` (standardmäßig `refit=True`). Für jeden Fold trainiert sie die **komplette Pipeline** neu, nicht nur KNN.

`verbose=3` macht diese Arbeit sichtbar: zuerst eine Zusammenfassung der Trainingsdurchläufe, danach Angaben zu Kandidat/Fold, Scores und Zeitmessung. Diese Scores sind **Validierungs-Scores**, keine abschließende Test-Accuracy. Setz `verbose=1` für eine kürzere Zusammenfassung. Mit `n_jobs=-1` arbeiten die verfügbaren CPU-Kerne parallel, deshalb können sich ausführliche Meldungen vermischen oder in der Kernel-Ausgabe des Editors erscheinen.

[Referenz: Ausgabedetails und erneutes Training bei GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [ ]:
# Verschachtelte Einstellungen untersuchen; knn__-Namen sprechen den Klassifikator in der Pipeline an.
knn_classifier_scaled.get_params()

In [ ]:
# Auszuprobierende Werte auflisten; das Grid Search evaluiert ihr kartesisches Produkt.
param_grid = {
    "knn__n_neighbors": [2, 3, 4, 5, 10],  # Anzahl der Nachbarn
    "knn__weights": ["uniform", "distance"],  # gleich oder nach Distanz gewichtete Stimmen
    "knn__p": [1, 2, 3],  # Minkowski-Distanz: 1 = Manhattan, 2 = euklidisch
}

# Dieselben Validierungs-Folds wie zuvor verwenden; mit verbose den Fortschritt ausführlich anzeigen.
gs = GridSearchCV(
    knn_classifier_scaled, param_grid, scoring="accuracy", cv=cv, verbose=3, n_jobs=-1
)

# Nur auf den Trainingsdaten trainieren: suchen, nach mittlerer CV-Accuracy auswählen, dann neu trainieren.
# Verstrichene Sekunden für diesen Durchlauf messen, kein allgemeiner Geschwindigkeits-Benchmark.
start = timer()
gs.fit(X_train, y_train)
end = timer()
gs_time = end - start

In [ ]:
gs_time

In [ ]:
# Mittlere Validierungs-Accuracy des besten Kandidaten, nicht die Test-Accuracy.
print("Best CV accuracy:", round(gs.best_score_, 3))

# Gewählte Hyperparameterwerte; das gelernte Modell steckt in best_estimator_.
print("Best parameters:", gs.best_params_)

In [ ]:
# Ausgabelogik an einer Stelle halten; sie nur für das endgültig ausgewählte Modell aufrufen.
def print_pretty_summary(name, best_model, y_test, y_pred_test):
    print(name)
    print("=======================")
    # Der letzte Pipeline-Schritt enthält den trainierten KNN-Klassifikator.
    model = best_model.steps[-1][1]
    print(f"n_neighbors: {model.n_neighbors}")
    print(f"weights: {model.weights}")
    print(f"p: {model.p}")
    print(f"algorithm: {model.algorithm}")

    accuracy = accuracy_score(y_test, y_pred_test)
    print(f"Final test accuracy: {accuracy:.3f}")
    return accuracy

In [ ]:
# Bereits auf ganz X_train neu trainiert, einschließlich Imputer, Encoder und Scaler.
knn_best = gs.best_estimator_
knn_best

In [ ]:
# Testdatensatz unberührt lassen, bis beide Suchen abgeschlossen sind.
# Zur Modellauswahl die Scores der Kreuzvalidierung vergleichen, nicht die Test-Scores.
print(f"Grid search best CV accuracy: {gs.best_score_:.3f}")

### RandomizedSearchCV

Als Alternative zum Grid Search können wir [RandomizedSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) aus sklearn verwenden. Das Random Search probiert nicht jede mögliche Kombination im Suchraum aus; stattdessen wählt es zufällig eine festgelegte Anzahl von Parameterkombinationen aus und evaluiert sie.

Verwende **denselben Raum aus 30 Kombinationen und dieselben Folds** wieder, damit sich im Vergleich nur die Suchstrategie ändert. `n_iter=10` zieht zehn verschiedene Kombinationen aus diesen Listen: **50 CV-Trainingsdurchläufe**, dann ein erneutes Training. `random_state=RSEED` macht diese Stichprobe wiederholbar. Die Logs verwenden dieselbe Einstellung `verbose=3` wie das Grid Search.

In [ ]:
# param_grid aus dem Grid Search wiederverwenden; weniger Kandidaten aus demselben Raum ziehen.
rs = RandomizedSearchCV(
    knn_classifier_scaled,
    param_grid,
    scoring="accuracy",
    cv=cv,
    n_iter=10,
    verbose=3,
    n_jobs=-1,
    random_state=RSEED,
)

# Nur auf X_train suchen, mit Vorverarbeitung pro Fold und einem abschließenden erneuten Training auf den Trainingsdaten.
start = timer()
rs.fit(X_train, y_train)
end = timer()
rgs_time = end - start

In [ ]:
# Diese mittlere Validierungs-Accuracy mit dem Grid Search vergleichen, nicht mit der Test-Accuracy.
print("Best CV accuracy:", round(rs.best_score_, 3))

# Beste Parameter
print("Best parameters:", rs.best_params_)

### Vor dem Testen auswählen

`best_score_` ist die mittlere CV-Accuracy des Gewinners; `best_params_` enthält seine Einstellungen; `best_estimator_` ist die komplette trainierte Pipeline. Weil die Suche auf die CV-Accuracy optimiert, kann der beste CV-Score optimistisch ausfallen; der beiseitegelegte Testdatensatz liefert eine separate abschließende Kontrolle.

Die Tabelle vergleicht die **beiden getunten KNN-Suchen**. Die früheren KNN/SGD-Beispiele veranschaulichen die Skalierung und sind keine Kandidaten in dieser abschließenden Auswahl. Wähl anhand der CV-Accuracy aus und ruf dann einmal `predict(X_test)` mit der ausgewählten Pipeline auf. Die Vorhersage verwendet die gelernte Vorverarbeitung wieder; sie trainiert das Modell nicht erneut.

In [ ]:
# Accuracy, Anzahl der ausprobierten Konfigurationen und verstrichene Zeit vergleichen.
searches = {"Grid search": gs, "Randomized search": rs}
comparison = pd.DataFrame(
    {
        "best_cv_accuracy": [gs.best_score_, rs.best_score_],
        "candidates": [len(gs.cv_results_["params"]), len(rs.cv_results_["params"])],
        "elapsed_seconds": [gs_time, rgs_time],
    },
    index=list(searches),
)
comparison

In [ ]:
# Auswählen, bevor die Testvorhersagen zu sehen sind; bei Gleichstand bleibt es beim ersten Eintrag im Dictionary.
selected_name = max(searches, key=lambda name: searches[name].best_score_)
selected_model = searches[selected_name].best_estimator_
# Erste und abschließende Evaluation auf den Testdaten: die trainierte Pipeline ohne neues Training anwenden.
y_pred_test = selected_model.predict(X_test)
print(f"Selected by CV accuracy: {selected_name}")
final_test_accuracy = print_pretty_summary(
    selected_name, selected_model, y_test, y_pred_test
)

## Interpretation

Die abschließende Evaluation verwendet das Modell, das die gewählte Suche bereits auf allen Trainingsdaten neu trainiert hat. Bei einem exakten Gleichstand in der CV wählt der Code das Grid Search, weil es im vordefinierten Dictionary zuerst steht.

Das Randomized Search evaluiert hier 10 der 30 aufgelisteten Kandidaten; das Grid Search evaluiert alle 30. Die Laufzeit hängt von Hardware und Caching ab, daher ist eine einzelne Zeitmessung kein Benchmark. Nutz den Test-Score nicht, um zurückzugehen und die Hyperparameter erneut zu tunen. Die Accuracy eignet sich für diesen einführenden Vergleich, beschreibt aber nicht jede Art von Klassifikationsfehler.

## Export für Notebook 3

Speichere nach Auswahl und abschließender Evaluation die trainierte Pipeline und eine kleine Metadatendatei. Dabei wird `selected_model` genau so wiederverwendet, wie es evaluiert wurde; es gibt kein neues Training und kein neues Hyperparameter-Tuning. Notebook 3 lädt diese Dateien in einem frisch gestarteten Kernel, um Vorhersagen für erfundene Requests zu machen.

Die Dateien bleiben im von Git ignorierten Verzeichnis `artifacts/`. Speichere die komplette Pipeline, einschließlich der Vorverarbeitung. [Referenz: scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
# Gelernte Kategorien aus der ausgewählten Pipeline auslesen; nicht erneut fitten.
fitted_preprocessor = selected_model.steps[0][1]
fitted_encoder = fitted_preprocessor.named_transformers_["categorical"].named_steps[
    "encoder"
]
# Namen der rohen Eingaben und Wertebereiche der Trainingsdaten behalten, damit Notebook 3 neue Requests validieren kann.
model_info = {
    "selected_search": selected_name,
    "input_features": list(X_train.columns),
    "numeric_features": col_scale,
    "categorical_features": col_encode,
    "known_categories": {
        column: categories.tolist()
        for column, categories in zip(
            col_encode, fitted_encoder.categories_, strict=True
        )
    },
    "numeric_training_ranges": X_train[col_scale].agg(["min", "max"]).to_dict(),
    "package_versions": {
        package: version(package)
        for package in ["scikit-learn", "numpy", "pandas", "joblib"]
    },
    "evaluation": {
        "cv_accuracy": float(searches[selected_name].best_score_),
        "test_accuracy": float(final_test_accuracy),
    },
}
# Trainiertes Modell dauerhaft speichern; die Metadaten beschreiben seine Eingaben und die vorhandene Evaluation.
joblib.dump(selected_model, artifact_dir / "titanic_pipeline.joblib")
(artifact_dir / "titanic_model_info.json").write_text(
    json.dumps(model_info, indent=2) + "\n"
)
print("Saved the fitted pipeline and metadata for notebook 3 in artifacts/.")

## Übersicht alle erzeugten Modelle


| # | Modell | Vorverarbeitung | Abschnitt | Trainingsläufe |
| --- | --- | --- | --- | --- |
| 1 | `DummyClassifier(strategy="most_frequent")` als Baseline | **keine**, Features werden komplett ignoriert | Baseline | 5 (CV) |
| 2 | KNN (Standard: k=5) | ohne Skalierung (`no_scaling_transformation`) | KNN: unskalierte Daten | 5 (CV) |
| 3 | KNN (k=5) | Standardisierung (`scaling_transformation`) | KNN: skalierte Daten | 5 (CV) |
| 4 | SGD (`loss="log_loss"`, also logistische Regression) | ohne Skalierung | SGD: unskalierte Daten | 5 (CV) |
| 5 | SGD (`log_loss`) | Standardisierung | SGD: skalierte Daten | 5 (CV) |
| 6 | KNN (k=5) | Min-Max-Skalierung (`normalization_transformation`) | Aufgabe 2, nur wenn gelöst | 5 (CV) |
| 7 | KNN mit **GridSearchCV**: alle 30 Kombinationen aus `n_neighbors`, `weights` und `p` | Standardisierung | Hyperparameter-Tuning | 30 × 5 = 150 + 1 Refit |
| 8 | KNN mit **RandomizedSearchCV**: 10 zufällige Kombinationen aus demselben Grid | Standardisierung | RandomizedSearchCV | 10 × 5 = 50 + 1 Refit |